In [1]:

import pandas as pd
import numpy as np
import re
from geopy.distance import geodesic

# Load file
file_path = "1742574481558_Dataset-ME-2025.xlsx"

middle_mile = pd.read_excel(file_path, sheet_name="Middle Mile")
tw_cost = pd.read_excel(file_path, sheet_name="T&W Cost")
ods_cost = pd.read_excel(file_path, sheet_name="ODS Cost")


In [2]:

routes = middle_mile[['Distributor Area', 'Origin Area', 'Origin Latitude', 'Longitude_x',
                      'Destination Area', 'Destination Latitude', 'Destination Longitude', 'Volume (CS)']].copy()

def safe_float(x):
    try:
        x = str(x).strip()
        if x.count(',') >= 1 and '.' not in x:
            x = x.replace(",", ".")
        return float(x)
    except:
        return np.nan

for col in ['Origin Latitude', 'Longitude_x', 'Destination Latitude', 'Destination Longitude']:
    routes[col] = routes[col].apply(safe_float)

routes.dropna(subset=['Origin Latitude', 'Longitude_x', 'Destination Latitude', 'Destination Longitude'], inplace=True)

def is_valid_coord(lat, lon):
    return -90 <= lat <= 90 and -180 <= lon <= 180

routes['Valid'] = routes.apply(lambda row: 
    is_valid_coord(row['Origin Latitude'], row['Longitude_x']) and 
    is_valid_coord(row['Destination Latitude'], row['Destination Longitude']), axis=1)

routes = routes[routes['Valid']].copy()

def calculate_distance(row):
    try:
        return geodesic((row['Origin Latitude'], row['Longitude_x']),
                        (row['Destination Latitude'], row['Destination Longitude'])).km
    except:
        return np.nan

routes['Distance_km'] = routes.apply(calculate_distance, axis=1)
routes.drop(columns='Valid', inplace=True)


In [3]:

def clean_currency(value):
    if pd.isnull(value):
        return np.nan
    try:
        clean = re.sub(r"[^\d]", "", str(value))
        if len(clean) > 9:
            clean = clean[-9:]
        return float(clean) / 1_000_000
    except:
        return np.nan

tw_cost = tw_cost[tw_cost['Total Cost'].notna()]
tw_cost["TnW_Cost_Million"] = tw_cost["Total Cost"].apply(clean_currency)
ods_cost = ods_cost[ods_cost['TOTAL COST'].notna()]
ods_cost["ODS_Cost_Million"] = ods_cost["TOTAL COST"].apply(clean_currency)


In [4]:

print("📌 Jumlah rute dengan koordinat valid:", len(routes))
print("📌 Rata-rata jarak antar rute:", round(routes['Distance_km'].mean(), 2), "km")

print("\n💰 Ringkasan Biaya T&W:")
print(tw_cost.groupby("Distributor Area")["TnW_Cost_Million"].sum().round(2))

print("\n🚛 Ringkasan Biaya ODS:")
print(ods_cost.groupby("Distributor Area")["ODS_Cost_Million"].sum().round(2))


📌 Jumlah rute dengan koordinat valid: 0
📌 Rata-rata jarak antar rute: nan km

💰 Ringkasan Biaya T&W:
Distributor Area
Bali Nusra    1761.00
Kalimantan    7977.89
Sulawesi      2834.90
Name: TnW_Cost_Million, dtype: float64

🚛 Ringkasan Biaya ODS:
Distributor Area
Bali Nusra    3712.82
Kalimantan    4463.28
Sulawesi      5645.95
Name: ODS_Cost_Million, dtype: float64
